# DestinE Data Lake (HDA): search, download, plot with EODAG

This notebook shows an end-to-end workflow:

1. Configure the **DestinE Data Lake (DEDL)** provider in **EODAG**
2. Discover / select a product type available through DEDL
3. Search a small time window
4. Download one product
5. Open the downloaded file (NetCDF/GRIB) and plot a sample map with `xarray`

**Prereqs**
- A DestinE Platform (DESP) account with access to the relevant collections
- Network access to DEDL endpoints from your environment


## 0) Install dependencies

In [ ]:
!pip install --user eodag

## 1) Configure EODAG for the DestinE Data Lake provider (`dedl`)

In [ ]:
import os
import getpass

# Option A (recommended in notebooks): prompt for credentials
# These are your DESP (DestinE Platform) username/password
username = os.environ.get("DESP_USERNAME") or input("DESP username: ").strip()
password = os.environ.get("DESP_PASSWORD") or getpass.getpass("DESP password: ")

# Configure the DEDL provider via environment variables.
# EODAG uses the convention: EODAG__<PROVIDER>__<SECTION>__<KEY>=...
os.environ["EODAG__DEDL__PRIORITY"] = "10"
os.environ["EODAG__DEDL__SEARCH__TIMEOUT"] = "60"
os.environ["EODAG__DEDL__AUTH__CREDENTIALS__USERNAME"] = username
os.environ["EODAG__DEDL__AUTH__CREDENTIALS__PASSWORD"] = password

print("EODAG DEDL provider configured via environment variables.")


## 2) Initialize EODAG and check that the `dedl` provider is available

In [ ]:
from eodag import EODataAccessGateway

dag = EODataAccessGateway()

providers = dag.available_providers()
print("Available providers:", providers)

assert "dedl" in providers, "Provider 'dedl' not available. Check your EODAG configuration."


## 3) Discover product types exposed by DEDL

In [ ]:
# This fetches product types from the provider
product_types = dag.list_product_types(provider="dedl")

# product_types can be a dict or a list depending on eodag version
if isinstance(product_types, dict):
    keys = sorted(product_types.keys())
else:
    keys = sorted([(pt.get("ID") or pt.get("id") or str(pt)) for pt in product_types])

# Prompt user for optional search terms (comma-separated)
raw_terms = input(
    "Enter search terms to filter product types (comma-separated, default is 'CAMS'): "
).strip()
if raw_terms:
    terms = [t.strip().upper() for t in raw_terms.split(",") if t.strip()]
else:
    terms = ["CAMS"]
for term in terms:
    search_keys = [k for k in keys if term in k.upper()]

print(f"Total product types from DEDL: {len(keys)}")
print(f"Candidate product types: {len(search_keys)}")

for i, k in enumerate(search_keys[:99], start=1):
    print(f"{i:2d}. {k}")

if not search_keys:
    PRODUCT_TYPE = None
    print("\nNo product type found.")
else:
    # Keep current PRODUCT_TYPE as default if it is in the list
    default_idx = 1

    while True:
        choice = input(
            f"\nPick a dataset [1-{len(search_keys)}] (default {default_idx}): "
        ).strip()

        if choice == "":
            selected_idx = default_idx
            break

        if choice.isdigit() and 1 <= int(choice) <= len(search_keys):
            selected_idx = int(choice)
            break

        print("Invalid choice. Enter a number in range or press Enter for default.")

    PRODUCT_TYPE = search_keys[selected_idx - 1]
    print("Selected PRODUCT_TYPE:", PRODUCT_TYPE)


## 4) Search a small time window and download one product

In [ ]:
import datetime as dt
from pathlib import Path

if not PRODUCT_TYPE:
    raise RuntimeError(
        "No product type found via DEDL. "
        "Inspect `search_keys` above and adjust filtering, or check access rights."
    )

# Choose a time window
DT_FORMAT = "%Y-%m-%d"
end = dt.datetime.now()
start = end - dt.timedelta(days=1000)
start = start.strftime(DT_FORMAT)
end = end.strftime(DT_FORMAT)

# Search.
# Note: EODAG expects ISO8601 strings or datetimes for start/end.
search_results = dag.search(
    provider="dedl",
    productType=PRODUCT_TYPE,
    start=start,
    end=end,
    items_per_page=20,
)
print("Results returned in this page:", len(search_results))

if len(search_results) == 0:
    raise RuntimeError(
        "No results returned. Try widening the time range or bbox (if used)."
    )

# Let user choose which product to download (first result is not always the desired data file).
print("\nAvailable products:")
for i, p in enumerate(search_results, start=1):
    props = getattr(p, "properties", {}) or {}
    pid = props.get("id", None) or getattr(p, "id", None) or "<no-id>"
    title = props.get("title", None) or "<no-title>"
    print(f"{i:2d}. {pid} | {title}")

default_idx = 1
while True:
    choice = input(
        f"\nPick a product to download [1-{len(search_results)}] (default {default_idx}): "
    ).strip()

    if choice == "":
        selected_idx = default_idx
        break

    if choice.isdigit() and 1 <= int(choice) <= len(search_results):
        selected_idx = int(choice)
        break

    print("Invalid choice. Enter a number in range or press Enter for default.")

product = search_results[selected_idx - 1]
props = getattr(product, "properties", {}) or {}
print("Selected product ID:", props.get("id", None) or getattr(product, "id", None))
print("Selected product title:", props.get("title", None))

# Download to a local directory
output_dir = Path("./downloads")
output_dir.mkdir(exist_ok=True)

downloaded_path = dag.download(product, output_dir=output_dir)
print("Downloaded to:", downloaded_path)


## 5) Open the downloaded file with `xarray` and inspect variables

In [ ]:
import xarray as xr

path = Path(downloaded_path)
if path.is_dir():
    # Search recursively and prioritize formats by expected readability
    ext_order = ["*.nc", "*.nc4", "*.grib", "*.grib2", "*.gb", "*.tif", "*.tiff"]
    candidates = []
    for pattern in ext_order:
        candidates.extend(sorted(path.rglob(pattern)))

    if not candidates:
        raise RuntimeError(f"Downloaded a directory but found no obvious data file inside: {path}")
else:
    candidates = [path]

ds = None
selected_file = None
errors = []

for data_file in candidates:
    suffix = data_file.suffix.lower()
    try:
        if suffix in {".nc", ".nc4"}:
            ds = xr.open_dataset(data_file)
        elif suffix in {".grib", ".grib2", ".gb"}:
            ds = xr.open_dataset(data_file, engine="cfgrib")
        elif suffix in {".tif", ".tiff"}:
            # Prefer xarray+rasterio engine; fallback to rioxarray if available
            try:
                ds = xr.open_dataset(data_file, engine="rasterio")
            except Exception:
                import rioxarray as rxr
                da = rxr.open_rasterio(data_file)
                ds = da.to_dataset(name=da.name or "raster")
        else:
            continue

        selected_file = data_file
        break
    except Exception as exc:
        errors.append(f"{data_file}: {repr(exc)}")

if ds is None:
    msg = "\n".join(errors[:10])
    raise RuntimeError(
        "Could not open any discovered candidate file. "
        f"Tried {len(candidates)} file(s). First errors:\n{msg}"
    )

print("Opened:", selected_file)
print(ds)
print("\nData variables:", list(ds.data_vars)[:50])


## 6) Plot a quick-look map

In [ ]:
import matplotlib.pyplot as plt

var_names = list(ds.data_vars)
for i, k in enumerate(var_names[:99], start=1):
    print(f"{i:2d}. {k}")

# choose a variable to plot
selected_idx = 1
while True:
    choice = input(
        f"\nPick a variable [1-{len(var_names)}] (default {selected_idx}): "
    ).strip()

    if choice == "":
        selected_idx = default_idx
        break

    if choice.isdigit() and 1 <= int(choice) <= len(var_names):
        selected_idx = int(choice)
        break

    print("Invalid choice. Enter a number in range or press Enter for default.")

chosen = var_names[selected_idx-1]
da = ds[chosen]
print("Chosen variable:", chosen)
print("Dims:", da.dims)

# Select the first timestep if a time-like dimension exists
for tdim in ["time", "valid_time", "forecast_time"]:
    if tdim in da.dims:
        da = da.isel({tdim: 0})
        break

# If multiple vertical levels, pick the first
for zdim in ["level", "lev", "height", "altitude"]:
    if zdim in da.dims:
        da = da.isel({zdim: 0})
        break

plt.figure()
da.plot()
plt.title(f"{chosen} (sample slice)")
plt.show()
